###Prerequisties

In [0]:
%sql
USE CATALOG wns24082026;

CREATE TABLE IF NOT EXISTS quickstart_schema.users (
    id INTEGER,
    name STRING,
    dob DATE,
    email STRING,
    gender STRING,
    country STRING,
    region STRING,
    city STRING,
    asset INTEGER,
    marital_status STRING
  );

DESCRIBE EXTENDED quickstart_schema.users


###Transaction 2 - Load users_001.csv into Delta table

In [0]:
df = spark.read.csv(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

###Transation 03 - Filter country

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

###Transaction 4 - Filter country = "United States"

In [0]:
df.filter(col("country")=="United States").write.saveAsTable("quickstart_schema.users", mode="OVERWRITE")

###List transactions on table

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "quickstart_schema.users")

delta_table.history().display()

###Versioning

####Pyspark

In [0]:
spark.read.option("versionAsOf",2).table("quickstart_schema.users").display()

####SQL

In [0]:
%sql
Select * from quickstart_schema.users version as of 2;

###Timestamp

In [0]:
spark.read.option("timestampAsOf","2026-08-28T06:10:15").table("quickstart_schema.users").display()

In [0]:
%sql
Select * from quickstart_schema.users timestamp as of "2026-08-28T06:12:47";

###Advantage - Restore

In [0]:
%sql
RESTORE Table quickstart_schema.users TO VERSION AS OF 1;

###Adding metadata/log for transaction

In [0]:
df.filter(col("country")=="India").write.option("userMetadata", "Filtering country for india").saveAsTable("quickstart_schema.users", mode="OVERWRITE")

In [0]:

from pyspark.sql.functions import col
import json
transaction_metadata ={
    "pipepine":"User Ingestion Pipeline",
    "Job Name" :"DailyUserIngestion",
    "trigger":"Scheduled"
}

df.filter(col("country") == "India").write.option(
    "userMetadata", json.dumps(transaction_metadata)
).saveAsTable("quickstart_schema.users", mode="OVERWRITE")